In [0]:
%sql
SELECT current_catalog() AS catalog_name, current_schema() AS schema_name;

CREATE VOLUME IF NOT EXISTS YellowTaxiData;

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# Get date 2 months back to ensure data is published
today = datetime.today() - relativedelta(months=3)
year = today.year
month = today.month

# Define URLs and paths
parquet_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"
raw_path = f"/Volumes/db_workspace_1/default/yellowtaxidata/{year}/raw_{month:02d}.parquet"
snappy_path = f"/Volumes/db_workspace_1/default/yellowtaxidata/{year}/snappy_{month:02d}.parquet"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(raw_path), exist_ok=True)

# Download the file
print(f"Downloading: {parquet_url}")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(parquet_url, headers=headers)

if response.status_code != 200:
    print(f"Failed to download: HTTP {response.status_code}")
elif not response.content[:4] == b'PAR1':
    print(f"URL did not return a valid Parquet file. Data for {year}-{month:02d} may not be published yet.")
else:
    # Save raw file
    with open(raw_path, "wb") as f:
        f.write(response.content)
    print(f"Raw file written to {raw_path}")

    # Re-save as snappy compressed parquet
    print("Converting to snappy compression...")
    df = pd.read_parquet(raw_path)
    df.to_parquet(snappy_path, compression="snappy")
    print(f"Snappy file written to {snappy_path}")

    # Load into Spark and verify
    print("Loading into Spark...")
    df_spark = spark.read.parquet(snappy_path)
    print(f"Row count: {df_spark.count():,}")
    df_spark.printSchema()
    df_spark.show(5)

    

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS bronze
    """)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.yellow_taxi
    USING DELTA
    AS
    SELECT
         *
        ,current_timestamp() AS ingested_at
        ,input_file_name()   AS source_file
        FROM parquet.`{snappy_path}`
        """)



    

In [0]:
like_pattern = f"%snappy_{month:02d}%"

already_loaded = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM bronze.yellow_taxi 
    WHERE source_file LIKE '{like_pattern}'
    """).collect()[0]['cnt']

print(f"Already loaded: {already_loaded}")

if already_loaded > 0:
    print(f"Data for {year}-{month} already exists, skipping.")
else:
    df_spark = df_spark.withColumn("ingested_at", current_timestamp()) \
           .withColumn("source_file", lit(snappy_path))
    df_spark.write.format("delta") \
        .mode("append") \
        .saveAsTable("bronze.yellow_taxi")
    print(f"Appended {year}-{month} to bronze.yellow_taxi")


In [0]:
%sql
drop schema bronze cascade

In [0]:
%sql
select source_file, count(*) from bronze.yellow_taxi
group by source_file